### Raw Data Cleaning

In [137]:
import pandas as pd

In [138]:
locations_df = pd.read_csv("../data/locations.csv")
providers_df = pd.read_csv("../data/providers.csv")
payers_df = pd.read_csv("../data/payers.csv")
patients_df = pd.read_csv("../data/patients.csv")
calls_df = pd.read_csv("../data/calls.csv")
appointments_df = pd.read_csv("../data/appointments.csv")

for name, df in [("locations", locations_df), ("providers", providers_df),
                  ("payers", payers_df), ("patients", patients_df),
                  ("calls", calls_df), ("appointments", appointments_df)]:
    print(name, df.shape)

locations (7, 4)
providers (27, 5)
payers (7, 3)
patients (25858, 4)
calls (47551, 6)
appointments (199076, 12)


In [139]:
def strip_text_columns(df):
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = df[col].str.strip()
        
    return df

locations_df = strip_text_columns(locations_df)
payers_df = strip_text_columns(payers_df)
patients_df = strip_text_columns(patients_df)
calls_df = strip_text_columns(calls_df)
providers_df = strip_text_columns(providers_df)
appointments_df = strip_text_columns(appointments_df)

def strip_column_titles(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    return df

locations_df = strip_column_titles(locations_df)
payers_df = strip_column_titles(payers_df)
patients_df = strip_column_titles(patients_df)
calls_df = strip_column_titles(calls_df)
providers_df = strip_column_titles(providers_df)
appointments_df = strip_column_titles(appointments_df)

### Appointments

In [140]:
appointments_df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2022-07-18,2022-07-10,P16,L5,PT1,PY7,Post-Op Check,False,Completed,115.51,0.55
1,A2,2022-07-18,2022-07-11,P16,L5,PT2,PY6,New Patient Consult,True,Completed,276.82,2.20
2,A3,2022-07-18,2022-07-18,P16,L5,PT2,PY6,Post-Op Check,False,Completed,113.27,0.64
3,A4,2022-07-18,2022-07-03,P16,L5,PT3,PY5,New Patient Consult,True,Completed,364.85,2.47
4,A5,2022-07-18,2022-07-11,P16,L5,PT2,PY6,Follow-up,False,Completed,159.54,0.85


In [141]:
appointments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199076 entries, 0 to 199075
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   appointment_id    199076 non-null  object 
 1   date              199076 non-null  object 
 2   booked_date       199076 non-null  object 
 3   provider_id       199076 non-null  object 
 4   location_id       199076 non-null  object 
 5   patient_id        199076 non-null  object 
 6   payer_id          199076 non-null  object 
 7   appointment_type  199076 non-null  object 
 8   is_new_patient    199076 non-null  bool   
 9   status            199076 non-null  object 
 10  revenue           197417 non-null  float64
 11  rvu               199076 non-null  float64
dtypes: bool(1), float64(2), object(9)
memory usage: 16.9+ MB


In [142]:
# Convert dates to datetime format
appointments_df['date'] = pd.to_datetime(appointments_df['date'], errors='coerce', format='%Y-%m-%d')
appointments_df['booked_date'] = pd.to_datetime(appointments_df['booked_date'], errors='coerce', format='%Y-%m-%d')

In [143]:
# Clean column names by stripping whitespace and converting to lowercase
# appointments_df.columns = appointments_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
# appointments_df.columns.tolist()

# Redundant now, swap to bulk processing for all dataframes

In [144]:
appointments_df.isna().sum()

appointment_id         0
date                   0
booked_date            0
provider_id            0
location_id            0
patient_id             0
payer_id               0
appointment_type       0
is_new_patient         0
status                 0
revenue             1659
rvu                    0
dtype: int64

In [145]:
# Find rows where revenue is null and status is Completed
# Meaning that the appointment was completed but revenue was not posted yet, might require followup if lost
blank_revenue_df = appointments_df[(appointments_df['revenue'].isna()) & (appointments_df['status'] == 'Completed')]
blank_revenue_df.shape

(1659, 12)

In [146]:
# No rows where revenue is 0 and status is Completed, so no need to follow up on those
appointments_df[(appointments_df['revenue'] == 0) & (appointments_df['status'] == 'Completed')]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [147]:
# Impossible dates, turn them into NaT but keep records for safekeeping rather than dropping completely
bad_dates = appointments_df['booked_date'] > appointments_df['date']
appointments_df.loc[bad_dates, 'booked_date'] = pd.NaT

In [148]:
appointments_df['booked_date'].value_counts()

booked_date
2026-08-14    280
2026-06-29    264
2026-08-17    264
2026-07-27    261
2026-07-22    261
             ... 
2022-08-06      2
2022-07-04      1
2022-07-08      1
2022-07-03      1
2022-08-08      1
Name: count, Length: 1523, dtype: int64

In [149]:
# Primary key validation
appointments_df['appointment_id'].nunique() == len(appointments_df)

False

In [150]:
# Drop duplicate rows based on the 'appointment_id' column
appointments_df.drop_duplicates(subset=['appointment_id'], keep='first', inplace=True)
appointments_df.shape # 211361 -> 210310

(198086, 12)

In [151]:
# Fine now
appointments_df['appointment_id'].nunique() == len(appointments_df)

True

In [152]:
# Provider validation, find appointments with provider_id that doesn't exist in providers_df
set(providers_df['provider_id'])
provider_error_appointments_df = appointments_df[~appointments_df['provider_id'].isin(set(providers_df['provider_id']))]
provider_error_appointments_df

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
907,A908,2022-10-10,2022-10-02,P99,L1,PT115,PY6,Follow-up,False,Completed,127.29,1.24
972,A973,2022-10-13,2022-10-09,P99,L1,PT97,PY6,Follow-up,False,Completed,171.09,1.36
973,A974,2022-10-13,2022-10-03,P99,L1,PT67,PY5,Injection/Procedure,False,Completed,555.89,3.95
1153,A1154,2022-10-24,2022-10-21,P99,L5,PT62,PY3,Injection/Procedure,False,No-Show,0.00,0.00
1256,A1257,2022-10-28,2022-10-15,P99,L5,PT12,PY4,Injection/Procedure,False,No-Show,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
194948,A194949,2026-08-24,2026-08-12,P99,L2,PT11980,PY1,Injection/Procedure,False,Completed,301.85,4.02
195455,A195456,2026-08-25,2026-08-19,P99,L6,PT25510,PY4,New Patient Consult,True,Completed,256.73,2.67
195615,A195616,2026-08-26,2026-08-14,P99,L3,PT25532,PY7,New Patient Consult,True,Cancelled,0.00,0.00
196097,A196098,2026-08-27,2026-08-21,P99,L7,PT25578,PY3,New Patient Consult,True,Completed,198.99,1.98


In [153]:
# P99 doesn't exist in providers_df, but it does exist in appointments_df. This is likely a data quality issue that needs to be addressed
set(appointments_df['provider_id'])

{'P1',
 'P10',
 'P11',
 'P12',
 'P13',
 'P14',
 'P15',
 'P16',
 'P17',
 'P18',
 'P19',
 'P2',
 'P20',
 'P21',
 'P22',
 'P23',
 'P24',
 'P25',
 'P26',
 'P27',
 'P3',
 'P4',
 'P5',
 'P6',
 'P7',
 'P8',
 'P9',
 'P99'}

In [154]:
# Some provider_ids in appointments_df do not exist in providers_df. Replace those with 'UNK' to indicate unknown provider
invalid_provider_mask = ~appointments_df['provider_id'].isin(set(providers_df['provider_id']))
appointments_df.loc[invalid_provider_mask, 'provider_id'] = 'UNK'

appointments_df[appointments_df['provider_id'] == 'UNK']

# Need to add a row to providers_df for the unknown provider so that the join works correctly later on
unknown_provider = pd.DataFrame([{
    'provider_id': 'UNK', 'provider_name': 'Unknown Provider',
    'specialty': 'Unknown', 'primary_location_id': None, 'hire_date': pd.NaT
}])
providers_df = pd.concat([providers_df, unknown_provider], ignore_index=True)
providers_df

,provider_id,provider_name,specialty,primary_location_id,hire_date
0,P1,Dr. James Nguyen,Orthopedic Surgery,L1,2023-05-24
1,P2,Dr. Maria Patel,Sports Medicine,L1,2022-08-27
2,P3,Dr. Robert Garcia,Spine Surgery,L1,2022-09-01
3,P4,Dr. David Rossi,Physical Medicine & Rehab,L1,2022-11-18
4,P5,Dr. Susan Chen,Orthopedic Surgery,L2,2022-11-03
5,P6,Dr. Michael Alvarez,Sports Medicine,L2,2023-01-31
6,P7,"Karen Bennett, PT",Physical Therapy,L2,2023-03-29
7,P8,Dr. Thomas Walsh,Pain Management,L2,2024-08-16
8,P9,Dr. Nancy Okafor,Orthopedic Surgery,L3,2022-08-22
9,P10,Dr. George Tanaka,Hand & Upper Extremity,L3,2023-06-12


In [155]:
appointments_df[~appointments_df['location_id'].isin(set(locations_df['location_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [156]:
appointments_df[~appointments_df['patient_id'].isin(set(patients_df['patient_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [157]:
appointments_df[~appointments_df['payer_id'].isin(set(payers_df['payer_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [158]:
appointments_df.shape

(198086, 12)

In [159]:
# Check for negative revenue values
appointments_df[appointments_df['revenue'] < 0]['revenue'].sum()

np.float64(0.0)

In [160]:
# Check for negative rvu values
appointments_df[appointments_df['rvu'] < 0]['rvu'].sum()

np.float64(0.0)

In [161]:
appointments_df['status'].value_counts()

status
Completed    165113
No-Show       23126
Cancelled      9847
Name: count, dtype: int64

In [162]:
# Revenue should only ever be present on Completed appointments
# These are basically appointments that never happened (canceled, no-show, etc.) so the revenue are just 0.00
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'].notna())]['revenue'].unique()

array([0.])

In [163]:
appointments_df['appointment_type'].value_counts()

appointment_type
Follow-up              80423
Physical Therapy       35228
Post-Op Check          33335
New Patient Consult    25857
Injection/Procedure    23243
Name: count, dtype: int64

In [164]:
# Check if the number of new patient consultations matches the number of new patients
len(appointments_df[appointments_df['appointment_type'] == 'New Patient Consult']) == len(appointments_df[appointments_df['is_new_patient'] == True])

True

In [165]:
# Checks if there's any non-completed appointments that have revenue greater than 0, which should not happen
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'] > 0)]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [166]:
# Makes sure that the revenue values are reasonable
appointments_df['revenue'].describe()

count    196433.000000
mean        151.848007
std         142.391047
min           0.000000
25%          83.790000
50%         116.310000
75%         165.120000
max         798.840000
Name: revenue, dtype: float64

In [167]:
appointments_df['rvu'].describe()

count    198086.000000
mean          1.204647
std           1.163330
min           0.000000
25%           0.620000
50%           0.870000
75%           1.260000
max           6.100000
Name: rvu, dtype: float64

In [ ]:
# 196433 (Revenue count) + 1653 = 198086 (RVU count), which is the total number of appointments in the dataframe.
# 1759 appointments are waiting on the calculated revenue, but they should all have rvu which works out in this case.
appointments_df['revenue'].isna().sum()

np.int64(1653)

### Patients

In [169]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25858 entries, 0 to 25857
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   patient_id        25858 non-null  object
 1   first_visit_date  25858 non-null  object
 2   referral_source   25858 non-null  object
 3   payer_id          25858 non-null  object
dtypes: object(4)
memory usage: 808.2+ KB


In [170]:
# Primary key validation
patients_df['patient_id'].nunique() == len(patients_df)

True

In [171]:
patients_df['first_visit_date'] = pd.to_datetime(patients_df['first_visit_date'], errors='coerce', format='%Y-%m-%d')

In [172]:
patients_df['first_visit_date'].isna().sum()

np.int64(0)

In [173]:
patients_df['first_visit_date'].value_counts()

first_visit_date
2026-08-31    51
2026-06-18    49
2026-07-15    48
2024-09-25    48
2025-08-29    48
              ..
2022-07-25     1
2022-07-22     1
2022-07-23     1
2022-07-20     1
2022-07-21     1
Name: count, Length: 1276, dtype: int64

In [174]:
patients_df.head()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2022-07-18,Insurance Directory,PY7
1,PT2,2022-07-18,SELF,PY6
2,PT3,2022-07-18,Online Search,PY5
3,PT4,2022-07-19,Friend/Family,PY1
4,PT5,2022-07-19,Online Search,PY3


In [175]:
# Inaccurate casing causing values to be counted separately when they are actually the same
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     9883
Self                   4913
Online Search          3673
Insurance Directory    3645
Friend/Family          2616
Physician referral      426
physician referral      419
self                    143
SELF                    140
Name: count, dtype: int64

In [176]:
# Clean and check
patients_df['referral_source'] = patients_df['referral_source'].str.strip().str.title()
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     10728
Self                    5196
Online Search           3673
Insurance Directory     3645
Friend/Family           2616
Name: count, dtype: int64

In [177]:
# No errors
invalid_payer_mask = ~patients_df['payer_id'].isin(set(payers_df['payer_id']))
patients_df[invalid_payer_mask]

,patient_id,first_visit_date,referral_source,payer_id


In [178]:
# Nothing dropped
patients_df.drop_duplicates()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2022-07-18,Insurance Directory,PY7
1,PT2,2022-07-18,Self,PY6
2,PT3,2022-07-18,Online Search,PY5
3,PT4,2022-07-19,Friend/Family,PY1
4,PT5,2022-07-19,Online Search,PY3
...,...,...,...,...
25853,PT25854,2026-09-05,Physician Referral,PY1
25854,PT25855,2026-09-05,Insurance Directory,PY3
25855,PT25856,2026-09-05,Friend/Family,PY7
25856,PT25857,2026-09-05,Self,PY2


### Payers

In [179]:
payers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   payer_id    7 non-null      object
 1   payer_name  7 non-null      object
 2   payer_type  7 non-null      object
dtypes: object(3)
memory usage: 300.0+ bytes


In [180]:
payers_df

,payer_id,payer_name,payer_type
0,PY1,Medicare,Medicare
1,PY2,Medi-Cal,Medicaid
2,PY3,Blue Cross,Commercial
3,PY4,Aetna,Commercial
4,PY5,Humana,Commercial
5,PY6,Self-Pay,Self-Pay
6,PY7,UnitedHealth,Commercial


In [181]:
# Primary key validation
payers_df['payer_id'].nunique() == len(payers_df)

True

In [182]:
payers_df['payer_type'].value_counts()

payer_type
Commercial    4
Medicare      1
Medicaid      1
Self-Pay      1
Name: count, dtype: int64

In [183]:
# Mostly insurance, but there are some self-pay patients as well
# Name and type matches, ex. Medicare with Medicare
payers_df[['payer_name', 'payer_type']]

,payer_name,payer_type
0,Medicare,Medicare
1,Medi-Cal,Medicaid
2,Blue Cross,Commercial
3,Aetna,Commercial
4,Humana,Commercial
5,Self-Pay,Self-Pay
6,UnitedHealth,Commercial


### Providers

In [184]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   provider_id          28 non-null     object
 1   provider_name        28 non-null     object
 2   specialty            28 non-null     object
 3   primary_location_id  27 non-null     object
 4   hire_date            27 non-null     object
dtypes: object(5)
memory usage: 1.2+ KB


In [185]:
providers_df

,provider_id,provider_name,specialty,primary_location_id,hire_date
0,P1,Dr. James Nguyen,Orthopedic Surgery,L1,2023-05-24
1,P2,Dr. Maria Patel,Sports Medicine,L1,2022-08-27
2,P3,Dr. Robert Garcia,Spine Surgery,L1,2022-09-01
3,P4,Dr. David Rossi,Physical Medicine & Rehab,L1,2022-11-18
4,P5,Dr. Susan Chen,Orthopedic Surgery,L2,2022-11-03
5,P6,Dr. Michael Alvarez,Sports Medicine,L2,2023-01-31
6,P7,"Karen Bennett, PT",Physical Therapy,L2,2023-03-29
7,P8,Dr. Thomas Walsh,Pain Management,L2,2024-08-16
8,P9,Dr. Nancy Okafor,Orthopedic Surgery,L3,2022-08-22
9,P10,Dr. George Tanaka,Hand & Upper Extremity,L3,2023-06-12


In [186]:
providers_df['hire_date'] = pd.to_datetime(providers_df['hire_date'], errors='coerce', format='%Y-%m-%d')

In [187]:
providers_df['hire_date'].isna().sum()

np.int64(1)

In [188]:
# Validate that all primary_location_id values in providers_df exist in locations_df
invalid_location_mask = ~providers_df['primary_location_id'].isin(set(locations_df['location_id']))
providers_df[invalid_location_mask]

,provider_id,provider_name,specialty,primary_location_id,hire_date
27,UNK,Unknown Provider,Unknown,None,NaT


In [189]:
# Primary key validation
providers_df['provider_id'].nunique() == len(providers_df)

True

In [190]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   provider_id          28 non-null     object        
 1   provider_name        28 non-null     object        
 2   specialty            28 non-null     object        
 3   primary_location_id  27 non-null     object        
 4   hire_date            27 non-null     datetime64[ns]
dtypes: datetime64[ns](1), object(4)
memory usage: 1.2+ KB


### Locations

In [191]:
locations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   location_id    7 non-null      object
 1   location_name  7 non-null      object
 2   city           7 non-null      object
 3   state          7 non-null      object
dtypes: object(4)
memory usage: 356.0+ bytes


In [192]:
locations_df.head()

,location_id,location_name,city,state
0,L1,Santa Clarita Office,Santa Clarita,CA
1,L2,Valencia Office,Valencia,CA
2,L3,Burbank Office,Burbank,CA
3,L4,Glendale Office,Glendale,CA
4,L5,Pasadena Office,Pasadena,CA


In [193]:
# Primary key validation
locations_df['location_id'].nunique() == len(locations_df)

True

In [194]:
# Small sample of locations_df, but all states should be CA. This is just a check for good practice.
len(locations_df[locations_df['state'] == 'CA']) == len(locations_df)

True

In [195]:
locations_df['city'].value_counts()

city
Santa Clarita    1
Valencia         1
Burbank          1
Glendale         1
Pasadena         1
Thousand Oaks    1
Northridge       1
Name: count, dtype: int64

In [196]:
locations_df['location_name'].value_counts()

location_name
Santa Clarita Office    1
Valencia Office         1
Burbank Office          1
Glendale Office         1
Pasadena Office         1
Thousand Oaks Office    1
Northridge Office       1
Name: count, dtype: int64

### Calls

In [197]:
calls_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47551 entries, 0 to 47550
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   call_id          47551 non-null  object
 1   date             47551 non-null  object
 2   location_id      47551 non-null  object
 3   call_type        47551 non-null  object
 4   outcome          47551 non-null  object
 5   handle_time_sec  47551 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 2.2+ MB


In [198]:
calls_df.head()

,call_id,date,location_id,call_type,outcome,handle_time_sec
0,C1,2022-07-01,L1,Reschedule,Booked,485
1,C2,2022-07-01,L1,New Patient Inquiry,Not Booked,397
2,C3,2022-07-01,L1,Reschedule,Booked,150
3,C4,2022-07-01,L1,General Inquiry,Info Only,87
4,C5,2022-07-01,L2,Reschedule,Booked,216


In [199]:
calls_df['date'] = pd.to_datetime(calls_df['date'], errors='coerce', format='%Y-%m-%d')

In [200]:
# Validate that all location_id values in calls_df exist in locations_df
invalid_location_mask = ~calls_df['location_id'].isin(set(locations_df['location_id']))
calls_df[invalid_location_mask]

,call_id,date,location_id,call_type,outcome,handle_time_sec


In [201]:
# Primary key validation
calls_df['call_id'].nunique() == len(calls_df)

True

In [202]:
calls_df['call_type'].value_counts()

call_type
New Patient Inquiry    16183
Reschedule             14655
General Inquiry         9510
Billing Question        7203
Name: count, dtype: int64

In [203]:
calls_df['outcome'].value_counts()

outcome
Booked        22641
Info Only     14795
Not Booked    10115
Name: count, dtype: int64

In [204]:
# Make sure that values makes sense, ex. handle_time_sec should be positive and not too high
calls_df['handle_time_sec'].describe()

count    47551.000000
mean       275.598221
std        136.076258
min         40.000000
25%        157.000000
50%        277.000000
75%        394.000000
max        510.000000
Name: handle_time_sec, dtype: float64